In [ ]:
"""
Gridworld — Q-learning with fully exploratory behavior policy
=============================================================
Learns Q(s, a) online using the off-policy TD update:
 
    Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]
 
The behavior policy is fully exploratory (eps = 1.0, fixed): every action
is chosen uniformly at random. Q-learning is off-policy, so the behavior
policy does NOT need to be greedy — the max over Q(s',·) in the target
decouples learning from behavior and still converges to Q*.
 
Alpha decays over episodes so that the Robbins-Monro conditions are
(approximately) satisfied and Q converges.
 
Greedy policy is extracted as  pi*(s) = argmax_a Q(s, a)  at the end.
"""
 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

<img src="images/03_example_vf1.png" width="300">

In [ ]:
# ------------------------------------------------------------------ #
# Environment                                                        #
# ------------------------------------------------------------------ #
N      = 25
GAMMA  = 0.9
ORANGE = {6, 7, 12, 16, 18, 21}
BLUE   = {17}
 
UP, RIGHT, DOWN, LEFT, STAY = 0, 1, 2, 3, 4
A_NAMES  = ["UP", "RIGHT", "DOWN", "LEFT", "STAY"]
A_ARROWS = ["↑",  "→",     "↓",    "←",    "○"  ]
 
def transition(s, a):
    """Returns (s', reward) for MDP"""
    r, c = s // 5, s % 5
    dr, dc = [(-1,0),(0,1),(1,0),(0,-1),(0,0)][a]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < 5 and 0 <= nc < 5):
        return s, -1                          # wall hit
    sp = 5 * nr + nc
    reward = +1 if sp in BLUE else (-1 if sp in ORANGE else 0)
    return sp, reward

In [ ]:
# ------------------------------------------------------------------ #
# Epsilon-greedy action selection                                    #
# ------------------------------------------------------------------ #
def epsilon_greedy(Q, s, eps, rng):
    """
    With probability eps    → pick a random action (explore)
    With probability 1-eps  → pick argmax_a Q(s,a)  (exploit)
    """
    if rng.random() < eps:
        return rng.integers(0, 5)
    return int(np.argmax(Q[s]))

In [ ]:
# ------------------------------------------------------------------ #
# Q-learning                                                         #
# ------------------------------------------------------------------ #
def q_learning(
    n_episodes  = 10000,
    alpha_0     = 0.5,      # initial learning rate
    alpha_min   = 0.01,     # floor for alpha
    decay_every = 500,      # decay period (episodes)
    decay_rate  = 0.90,     # multiplicative decay factor
    seed        = 42,
):
    rng = np.random.default_rng(seed)
    Q   = np.zeros((N, 5))          # action-value table
 
    eps   = 1.0              # fully exploratory: fixed, never decayed
    alpha = alpha_0
 
    ep_returns  = []   # sum of rewards per episode (for plotting)
    ep_lengths  = []   # steps per episode
 
    for ep in range(n_episodes):
        if (ep%2000 == 0):
            print(f"episode: {ep} done")
 
        # ---- decay schedule (alpha only — eps is fixed at 1.0) ----
        if ep > 0 and ep % decay_every == 0:
            alpha = max(alpha * decay_rate, alpha_min)
 
        # ---- start of episode: pick random non-blue starting state --
        s = rng.integers(0, N)
        a = epsilon_greedy(Q, s, eps, rng)
 
        total_r = 0
        steps   = 0
        max_steps = 300   # cap episode length (continuing task)
 
        # ---- Q-learning update loop --------------------------------
        while steps < max_steps:
            sp, r = transition(s, a)
 
            # Core Q-learning update: target uses max over Q(s',·)
            td_error  = None   # off-policy target
            Q[s, a]  = Q[s,a] - None
 
            total_r += r
            steps   += 1
            s = sp
            a = epsilon_greedy(Q, s, eps, rng)   # eps=1.0 → always random
 
        ep_returns.append(total_r)
        ep_lengths.append(steps)
 
    return Q, ep_returns, ep_lengths

In [ ]:
# ------------------------------------------------------------------ #
# Run Q-learning                                                     #
# ------------------------------------------------------------------ #
print("Running Q-learning …")
Q, ep_returns, ep_lengths = q_learning()
print("Done!")

In [ ]:
# ------------------------------------------------------------------ #
# Greedy policy                                                      #
# ------------------------------------------------------------------ #
greedy_policy = np.argmax(Q, axis=1)
 
print("\nLearned Q-table (max Q per state, reshaped 5x5):")
print(np.array2string(np.max(Q, axis=1).reshape(5, 5),
                      formatter={"float_kind": lambda x: f"{x:5.1f}"}))
 
print("\nGreedy policy (arrows):")
grid_arrows = np.array([A_ARROWS[a] for a in greedy_policy]).reshape(5, 5)
for row in grid_arrows:
    print("  " + "   ".join(row))

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation policy                                               #
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 1, figsize=(4, 4))

ax = axes
V = np.max(Q, axis=1).reshape(5, 5)
 
# Background colour per cell type
cell_colors = np.zeros((5, 5))
for s in range(N):
    r, c = s // 5, s % 5
    if s in BLUE:
        cell_colors[r, c] = 2
    elif s in ORANGE:
        cell_colors[r, c] = 1
# 
cmap_bg = ListedColormap(["white", "#FF7F27", "#00A2E8"])
ax.imshow(cell_colors, cmap=cmap_bg, vmin=0, vmax=2, alpha=0.9)
 
dx = { UP:(0,-.35), RIGHT:(.35,0), DOWN:(0,.35), LEFT:(-.35,0), STAY:(0,0) }
for s in range(N):
    r, c  = s // 5, s % 5
    a     = greedy_policy[s]
    ddx, ddy = dx[a]
    if a == STAY:
        ax.plot(c, r, 'o', color="green", ms=10)
    else:
        ax.annotate("", xy=(c+ddx, r+ddy), xytext=(c-ddx, r-ddy),
                    arrowprops=dict(arrowstyle="-|>", color="green", lw=1.8))
# 
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_title("Greedy policy  π*(s) = argmax Q(s,·)")
 
plt.tight_layout()

In [ ]:
# ------------------------------------------------------------------ #
# Q-values (per action)                                              #
# ------------------------------------------------------------------ #
print("        a_1   a_2   a_3   a_4   a_5")
for i in range(Q.shape[0]):
    if (i%5 == 0):
        print(37*'-')
    print(f"s_{i+1:02d}:", np.array2string(Q[i,:], formatter={"float_kind": lambda x: f"{x:5.2f}"}))

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation returns                                              #
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 1, figsize=(7, 4))

ax = axes
window = 500
ax.plot(ep_returns, color="tab:blue", linewidth=1.0)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("Return")
ax.grid(True, alpha=0.3)
 
plt.tight_layout()    